# TF-IDF & Word Embeding

TF-IDF (Term Frequency–Inverse Document Frequency) dan word embedding adalah dua teknik representasi teks yang berbeda dalam pemrosesan bahasa alami. TF-IDF bekerja dengan menghitung bobot kata berdasarkan seberapa sering kata muncul dalam sebuah dokumen (TF) dibandingkan dengan seberapa jarang kata tersebut muncul di seluruh koleksi dokumen (IDF), sehingga cocok untuk mengekstraksi kata-kata penting secara statistik. Sementara itu, word embedding (seperti Word2Vec, GloVe, atau FastText) merepresentasikan kata ke dalam vektor berdimensi rendah yang menangkap hubungan semantik dan konteks antar kata, sehingga kata-kata dengan makna serupa memiliki representasi vektor yang dekat. Dengan kata lain, TF-IDF lebih fokus pada frekuensi dan kepentingan kata dalam dokumen, sedangkan word embedding lebih menekankan pemahaman makna dan hubungan antar kata dalam ruang vektor.

## Library

In [1]:
!pip install plotly
!pip install --upgrade gensim

In [2]:
from gensim.models import Word2Vec, FastText
import pandas as pd
import re

from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer

from matplotlib import pyplot as plt
import plotly.graph_objects as go

import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [3]:
pta_df_wfq = pd.read_excel("pta_word_frequency.xlsx", engine='openpyxl')
pta_df_wfq.head(10)

,kata,jumlah
0,pengaruh,5560
1,kerja,5394
2,teliti,4385
3,variabel,3677
4,usaha,2538
5,signifikan,2494
6,uji,2370
7,karyawan,2273
8,nilai,1912
9,hasil,1788


In [4]:
df = pd.read_csv('pta_abstrak.csv')

In [5]:
clean_txt = []
for w in range(len(df.abstrak_normal)):
    # ubah NaN jadi string kosong
    desc = str(df['abstrak_normal'][w]).lower()

    # remove punctuation
    desc = re.sub('[^a-zA-Z]', ' ', desc)

    # remove tags
    desc = re.sub("&lt;/?.*?&gt;", " ", desc)

    # remove digits and special chars
    desc = re.sub("(\\d|\\W)+", " ", desc)

    clean_txt.append(desc.strip())

df['clean'] = clean_txt
df.head()

,abstrak_normal,clean
0,abstrak aliyah pengaruh faktor latih kembang p...,abstrak aliyah pengaruh faktor latih kembang p...
1,tuju teliti persepsi band association langgan ...,tuju teliti persepsi band association langgan ...
2,NaN,nan
3,aplikasi nyata manfaat teknologi informasi kom...,aplikasi nyata manfaat teknologi informasi kom...
4,abstrak teliti metode kuantitatif tekan uji hi...,abstrak teliti metode kuantitatif tekan uji hi...


## TF-IDF

In [6]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['clean'])

# ubah ke DataFrame biar keliatan
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)

In [21]:
print("\nTF-IDF shape:", tfidf_df.shape)
tfidf_df


TF-IDF shape: (1031, 5445)


,aa,aaaamanahsyariah,ab,abadi,abah,abai,abal,abas,abc,abdu,...,zakiyatus,zaman,zan,zavgren,zebra,zmiewski,zmijewski,zmijewskiterhadap,zulkifli,zulkiflimsi
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1026,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1027,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1028,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1029,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Word Embeding

In [8]:
corpus = []
for col in df.clean:
   word_list = col.split(" ")
   corpus.append(word_list)

#show first value
corpus[0:1]

#generate vectors from corpus
model = Word2Vec(corpus, min_count=1, vector_size = 56)

In [9]:
# explore embeddings using cosine similarity
print("Kata mirip dengan 'penelitian':")
print(model.wv.most_similar('penelitian', topn=5))

print("\nKata mirip dengan 'data':")
print(model.wv.most_similar('data', topn=5))

# contoh cosmul
print("\nCosmul (penelitian + sistem - data):")
print(model.wv.most_similar_cosmul(positive=['penelitian', 'sistem'], negative=['data'], topn=5))

# doesnt_match: cari kata yang tidak sesuai konteks
print("\nKata yang tidak cocok dalam ['penelitian', 'data', 'sistem', 'informasi']:")
print(model.wv.doesnt_match("penelitian data sistem informasi".split()))

# save embeddings
filename = 'pta_embeddings.txt'
model.wv.save_word2vec_format(filename, binary=False)
print(f"\nEmbeddings disimpan ke {filename}")


Kata mirip dengan 'penelitian':
[('jadi', 0.970125675201416), ('deskripsi', 0.9594250917434692), ('promosional', 0.9499030709266663), ('eksploratif', 0.9493480324745178), ('landas', 0.9484647512435913)]

Kata mirip dengan 'data':
[('olah', 0.953402578830719), ('sekunder', 0.9472562074661255), ('kumpul', 0.9448931217193604), ('primer', 0.9434851408004761), ('gun', 0.9377446174621582)]

Cosmul (penelitian + sistem - data):
[('kub', 1.301365852355957), ('gula', 1.2826875448226929), ('sejahtera', 1.2825367450714111), ('eksternal', 1.2792693376541138), ('sumberrejo', 1.277072548866272)]

Kata yang tidak cocok dalam ['penelitian', 'data', 'sistem', 'informasi']:
data

Embeddings disimpan ke pta_embeddings.txt


In [10]:
class MyTokenizer:
    def fit_transform(self, texts):
        # Tokenisasi sederhana: lowercase + split
        return [str(text).lower().split() for text in texts]

class MeanEmbeddingVectorizer:
    def __init__(self, word2vec_model):
        self.word2vec = word2vec_model
        # Perbaikan: gunakan vector_size (Gensim ≥ 4.0)
        self.dim = word2vec_model.wv.vector_size

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_tokenized = MyTokenizer().fit_transform(X)
        embeddings = []
        for words in X_tokenized:
            # Ambil vektor hanya untuk kata yang ada di vocab
            valid_vectors = [
                self.word2vec.wv[word] for word in words
                if word in self.word2vec.wv
            ]
            if valid_vectors:
                embeddings.append(np.mean(valid_vectors, axis=0))
            else:
                embeddings.append(np.zeros(self.dim))
        return np.array(embeddings)

    def fit_transform(self, X, y=None):
        return self.transform(X)

In [11]:
df.shape

(1031, 2)

In [12]:
mean_embedding_vectorizer = MeanEmbeddingVectorizer(model)
mean_embedded = mean_embedding_vectorizer.fit_transform(df['clean'])

In [13]:
df['array']=list(mean_embedded)

In [14]:
df.head(5)

,abstrak_normal,clean,array
0,abstrak aliyah pengaruh faktor latih kembang p...,abstrak aliyah pengaruh faktor latih kembang p...,"[-0.78004766, 0.2559402, 0.4899151, -0.1596653..."
1,tuju teliti persepsi band association langgan ...,tuju teliti persepsi band association langgan ...,"[-0.8211994, 0.81821376, 0.37096956, -0.467466..."
2,NaN,nan,"[-0.014413638, 0.009600126, 0.0088578295, 0.00..."
3,aplikasi nyata manfaat teknologi informasi kom...,aplikasi nyata manfaat teknologi informasi kom...,"[-0.57507205, 0.4152917, 0.27015114, -0.385189..."
4,abstrak teliti metode kuantitatif tekan uji hi...,abstrak teliti metode kuantitatif tekan uji hi...,"[-0.7094513, 0.45650932, 0.47011524, 0.0166120..."


In [15]:
df['embedding_length'] = df['array'].str.len()

In [16]:
print(df['embedding_length'])

0       56
1       56
2       56
3       56
4       56
        ..
1026    56
1027    56
1028    56
1029    56
1030    56
Name: embedding_length, Length: 1031, dtype: int64


In [17]:
df.shape

(1031, 4)

In [20]:
num_features = len(df['array'].iloc[0])  # asumsi semua list punya panjang sama
columns = [f'f{i+1}' for i in range(num_features)]

# Inisialisasi dictionary untuk menampung data per kolom
data_dict = {col: [] for col in columns}

# Looping setiap baris di kolom 'embedding'
for embedding_list in df['array']:
    for i, value in enumerate(embedding_list):
        data_dict[f'f{i+1}'].append(value)

# Buat DataFrame dari dictionary
embedding_df = pd.DataFrame(data_dict)

In [19]:
embedding_df

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f47,f48,f49,f50,f51,f52,f53,f54,f55,f56
0,-0.780048,0.255940,0.489915,-0.159665,0.524890,0.220230,0.321184,-0.647398,-0.139354,-0.745447,...,-0.140312,0.691551,0.561251,0.243844,-0.101491,0.574063,-0.168173,0.619659,-0.105297,-0.722767
1,-0.821199,0.818214,0.370970,-0.467466,0.591698,-0.053629,0.426945,-0.471364,-0.064859,-0.618603,...,0.163997,-0.043986,-0.124192,0.253013,0.179889,0.543135,0.243381,0.713388,0.184684,-0.034495
2,-0.014414,0.009600,0.008858,0.006671,-0.007198,0.017463,0.004589,-0.000090,0.001711,0.003228,...,-0.011677,-0.013732,-0.006864,-0.004532,0.004385,-0.003233,0.015214,-0.001837,-0.007297,0.005300
3,-0.575072,0.415292,0.270151,-0.385190,0.528811,0.161737,0.377603,-0.504310,0.013026,-0.326801,...,-0.017239,0.114214,0.085472,0.264248,-0.034670,0.396175,0.068002,0.654628,0.132913,-0.269046
4,-0.709451,0.456509,0.470115,0.016612,0.720554,0.468291,0.465154,-0.750870,-0.120305,-1.297483,...,0.038180,0.811701,0.322323,0.580477,-0.501157,0.565366,0.069777,0.738400,-0.085739,-0.803870
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1026,-0.508679,0.292436,0.056535,-0.067953,0.228190,0.044267,0.302147,-0.519348,-0.210076,-0.641326,...,0.203158,0.419612,-0.170325,0.298605,-0.073364,0.353185,0.300068,0.505158,0.020359,-0.092272
1027,-0.405399,0.013446,0.358071,0.251381,0.624648,0.296721,0.519837,-0.796731,-0.145358,-1.240324,...,-0.095443,1.250402,0.523080,0.601003,-0.606716,0.478210,0.015677,0.437794,-0.108863,-1.083021
1028,-0.615385,0.580129,0.266329,-0.079532,0.595972,0.279345,0.547993,-0.596586,-0.149240,-1.141819,...,0.261172,0.511233,-0.190559,0.542525,-0.347486,0.495145,0.413520,0.734767,0.081510,-0.343902
1029,-1.026976,0.539842,0.397087,-0.305435,0.988281,0.351483,0.584744,-0.638531,0.023028,-0.426570,...,0.016435,0.123900,-0.034524,0.420352,-0.191106,0.294982,0.066935,0.915964,0.295197,-0.218674


In [22]:
embedding_df.shape

(1031, 56)